First look: ดูโครงสร้าง ช่วงเวลา หน่วย และค่าหาย ไม่วิเคราะห์

In [1]:
import requests
import pandas as pd
import truststore; truststore.inject_into_ssl()     # ดูตารางแก้ปัญหา: SSLError

URL = "http://air4thai.pcd.go.th/services/getNewAQI_JSON.php"
r = requests.get(URL, timeout=30)
r.status_code, r.headers.get("Content-Type")

(200, 'application/json; charset=UTF-8')

In [2]:
data = r.json()
print(type(data), list(data.keys()) if isinstance(data, dict) else len(data))
stations = data["stations"]          # ถ้า key ไม่ชื่อนี้ ให้แก้ตามที่เห็นจาก cell ก่อนหน้า
print(len(stations))
stations[0]                          # ดูโครงสร้างของสถานีแรกทั้งก้อน

<class 'dict'> ['stations']
179


{'stationID': '119t',
 'nameTH': 'สวนสาธารณะธารา',
 'nameEN': 'Thara Public Park',
 'areaTH': 'ต.ปากน้ำ อ.เมือง, กระบี่',
 'areaEN': 'Pak Nam Subdistrict, Mueang District, Krabi',
 'stationType': 'GROUND',
 'lat': '8.0506237',
 'long': '98.9180489',
 'forecast': [],
 'AQILast': {'date': '2026-11-20',
  'time': '20:00',
  'PM25': {'color_id': '1', 'aqi': '9', 'value': '5.5'},
  'PM10': {'color_id': '0', 'aqi': '-1', 'value': '-1'},
  'O3': {'color_id': '1', 'aqi': '16', 'value': '22'},
  'CO': {'color_id': '1', 'aqi': '2', 'value': '0.30'},
  'NO2': {'color_id': '1', 'aqi': '1', 'value': '2'},
  'SO2': {'color_id': '1', 'aqi': '1', 'value': '2'},
  'AQI': {'color_id': '1', 'aqi': '16', 'param': 'O3'}}}

In [3]:
df_now = pd.json_normalize(stations)
print(df_now.shape)
df_now.columns.tolist()              # ชื่อคอลัมน์ที่ซ้อนกันจะต่อด้วยจุด เช่น "<key>.<subkey>"

(179, 32)


['stationID',
 'nameTH',
 'nameEN',
 'areaTH',
 'areaEN',
 'stationType',
 'lat',
 'long',
 'forecast',
 'AQILast.date',
 'AQILast.time',
 'AQILast.PM25.color_id',
 'AQILast.PM25.aqi',
 'AQILast.PM25.value',
 'AQILast.PM10.color_id',
 'AQILast.PM10.aqi',
 'AQILast.PM10.value',
 'AQILast.O3.color_id',
 'AQILast.O3.aqi',
 'AQILast.O3.value',
 'AQILast.CO.color_id',
 'AQILast.CO.aqi',
 'AQILast.CO.value',
 'AQILast.NO2.color_id',
 'AQILast.NO2.aqi',
 'AQILast.NO2.value',
 'AQILast.SO2.color_id',
 'AQILast.SO2.aqi',
 'AQILast.SO2.value',
 'AQILast.AQI.color_id',
 'AQILast.AQI.aqi',
 'AQILast.AQI.param']

1. หนึ่งแถวของ df_now คืออะไร (หนึ่งสถานี ณ เวลาล่าสุด?) <br>
*ANSWER:* หนึ่งแถวคือข้อมูลของ 1 สถานีตรวจวัด ณ เวลารายงานล่าสุด โดยทั้งหมดมี 178 แถว <br> <br>
2. คอลัมน์ใดเก็บรหัสสถานี ชื่อ พิกัด วันที่ และเวลา <br>
*ANSWER:* stationID, nameTH, nameEN, lat, long, AQILast.date, AQILast.time <br> <br>
3. ค่า PM2.5 อยู่คอลัมน์ไหน มีทั้ง ค่าความเข้มข้น และ ค่า AQI หรือไม่ 
หน่วยคืออะไร <br>
*ANSWER:* มีทั้งคู่ ค่าความเข้มข้นอยู่คอลัมน์ AQILast.PM25.value ค่าความเข้มข้น PM2.5 ของ Air4Thai มีหน่วยเป็น µg/m³ หรือไมโครกรัมต่อลูกบาศก์เมตร แล้วค่า AQI คือ AQILast.PM25.aqi ระดับสีที่ AQILast.PM25.color_id <br> <br>
4. ค่าตัวเลขถูกเก็บเป็นข้อความ (object) หรือตัวเลข มีค่าแปลก เช่น ค่าลบ หรือ "-" แทนค่าหายหรือไม่ <br>
*ANSWER:* ค่าตัวเลขถูกเก็บเป็นข้อความ ไม่มีค่าที่แปลงเป็นตัวเลขไม่ได้ (0 แถว เช่น "-"), แต่พบค่าติดลบ -1 จำนวน 2 แถว ใน AQILast.PM25.value (min = -1.0) ซึ่งใช้แทนค่าหาย/ไม่มีข้อมูล (เหมือนที่เห็นใน 'PM10': {'aqi': '-1', 'value': '-1'} ของ stations[0]) <br>

In [4]:
pm = pd.to_numeric(df_now["AQILast.PM25.value"], errors="coerce")
print("แปลงเป็นตัวเลขไม่ได้:", pm.isna().sum())
print("ค่าติดลบ:", (pm < 0).sum())
pm.describe()

แปลงเป็นตัวเลขไม่ได้: 0
ค่าติดลบ: 1


count    179.000000
mean      14.904469
std        5.699673
min       -1.000000
25%       11.750000
50%       15.400000
75%       18.300000
max       35.900000
Name: AQILast.PM25.value, dtype: float64

In [5]:
import requests
import pandas as pd
import truststore; truststore.inject_into_ssl()     # ดูตารางแก้ปัญหา: SSLError

hourly = pd.read_csv("../data/sample/pm25_hourly_sample.csv", parse_dates=["ts"])
st = pd.read_csv("../data/sample/stations_sample.csv")

# ใช้เฉพาะช่วงสำรวจ 2563–2566 (ค.ศ. 2020–2023) ห้ามดูช่วงทดสอบ
explore = hourly[hourly["ts"] < "2026-11-01"]

print(explore.shape, explore["station_id"].nunique(), "สถานี")
print(explore["ts"].min(), "→", explore["ts"].max())
print(explore["ts"].diff().value_counts().head(3))   # ความถี่ของข้อมูล (ควรเป็น 1 ชั่วโมง)
explore.isna().mean().round(3)

(260592, 8) 178 สถานี
2026-09-01 00:00:00 → 2026-10-31 23:00:00
ts
0 days 01:00:00       260414
-61 days +01:00:00       177
Name: count, dtype: int64


station_id    0.000
ts            0.000
pm25          0.073
pm10          0.647
o3            0.849
co            0.809
no2           0.792
so2           0.828
dtype: float64

In [6]:
# สถานีที่ค่า PM2.5 หายมากที่สุด 10 อันดับ (ดูว่าค่าหายอยู่ตรงไหน ยังไม่ตัดสินว่าทำไม)
(explore.groupby("station_id")["pm25"]
        .apply(lambda s: s.isna().mean())
        .sort_values(ascending=False)
        .head(10))

station_id
o59    1.000000
o64    1.000000
o68    1.000000
o67    1.000000
o29    1.000000
18t    0.837432
74t    0.770492
76t    0.748634
26t    0.695355
75t    0.621585
Name: pm25, dtype: float64

1. **ช่วงเวลาและโครงสร้าง:** ไฟล์ตัวอย่างมีเฉพาะข้อมูล `2026-09-01` ถึง `2026-11-17` (`333,216` แถว, `178` สถานี) ไม่มีช่วงปี 2563–2566 จึงกรอง `ts < "2026-11-01"` เป็นช่วงสำรวจ (`260,592` แถว)
2. **ความถี่:** พบว่า `ts.diff()` ส่วนใหญ่คือ `1 ชั่วโมง` (`0 days 01:00:00`) ส่วนค่าติดลบ (`177` ครั้ง) เกิดจากรอยต่อระหว่าง `178` สถานี ควรตรวจสอบต่อว่ามีชั่วโมงที่ขาดหายระหว่างสถานีหรือไม่
3. **ค่าหายรายคอลัมน์:** พบว่า `pm25` หาย `7.3%` ส่วน `pm10` (`64.7%`) และก๊าซอื่น (`79%–85%`) หายสูงมาก ควรตรวจสอบต่อว่าหลายสถานีไม่มีเซนเซอร์วัดสารเหล่านี้หรือไม่
4. **สถานีที่ `pm25` หายสูงสุด:** พบ `5` สถานี (`o59`, `o64`, `o68`, `o67`, `o29`) ค่าหาย `100%` และอีก `5` สถานี (`18t`, `74t`, `76t`, `26t`, `75t`) หาย `62%–84%` ควรตรวจสอบต่อก่อนนำไปคัดกรองหรือวิเคราะห์